In [34]:
import random
import tensorflow as tf
import keras
import numpy as np
import pandas as pd
import copy

from keras import layers
from keras import models
from keras import initializers

print(f"tensorflow version: {tf.__version__}, keras version: {keras.__version__}, numpy version: {np.__version__}")

tensorflow version: 2.16.1, keras version: 3.3.3, numpy version: 1.24.3


In [77]:
nn_model = models.Sequential([
            layers.Input(shape=(4,)),
            layers.Dense(4, activation=keras.activations.leaky_relu,
                         kernel_initializer=initializers.RandomNormal(stddev=0.05),
                         bias_initializer=initializers.RandomNormal(stddev=0.05)),
            layers.Dense(1, activation=keras.activations.tanh,
                         kernel_initializer=initializers.RandomNormal(stddev=0.05),
                         bias_initializer=initializers.RandomNormal(stddev=0.05))
        ])

In [6]:
print(nn_model)

<Sequential name=sequential, built=True>


In [78]:
loss_fn = keras.losses.MeanSquaredError()
optimizer = keras.optimizers.SGD(learning_rate=0.001)

In [8]:
print(loss_fn)

In [26]:
y_pred = tf.constant([[1., 1.]], dtype=tf.float32)
y_actual = tf.constant([[5., 5.]], dtype=tf.float32)

loss_fn(y_pred, y_actual)

<tf.Tensor: shape=(), dtype=float32, numpy=16.0>

In [79]:
x_0 = tf.constant([[1., 1., 1., 1.]], dtype=tf.float32)
x_1 = tf.constant([[-1., 1., -1., 1.]], dtype=tf.float32)

y_0 = nn_model(x_0)
y_1 = nn_model(x_1)

print(f"Test: {x_0} -> {y_0}, {x_1} -> {y_1}")

Test: [[1. 1. 1. 1.]] -> [[-0.04264859]], [[-1.  1. -1.  1.]] -> [[-0.02626758]]


In [80]:
y_expected_0 = tf.constant([[0.5]], shape=(1, 1), dtype=tf.float32)
y_expected_1 = tf.constant([[-0.5]], shape=(1, 1), dtype=tf.float32)

#print(nn_model.trainable_weights)

#nn_model_copy = copy.deepcopy(nn_model)

In [81]:
learning_rate = 0.01

def adjust_weights(x_input, y_exp):
    with tf.GradientTape() as tape:
        y_pred = nn_model(x_0)
        gradients = tape.gradient(y_pred, nn_model.trainable_weights)

    for i, gradient in enumerate(gradients):
        weight = nn_model.trainable_weights[i]
        weight.assign_add(learning_rate * tf.reshape(y_exp - y_pred, shape=(1,)) * gradient)
        

In [38]:
print(nn_model.trainable_weights)

[<KerasVariable shape=(4, 4), dtype=float32, path=sequential/dense/kernel>, <KerasVariable shape=(4,), dtype=float32, path=sequential/dense/bias>, <KerasVariable shape=(4, 1), dtype=float32, path=sequential/dense_1/kernel>, <KerasVariable shape=(1,), dtype=float32, path=sequential/dense_1/bias>]


In [68]:
nn_model.get_weights()

[array([[-0.00824536, -0.0388292 , -0.00148987, -0.04924238],
        [ 0.02027069, -0.02949522, -0.06782671,  0.00613663],
        [-0.08612888,  0.0505438 , -0.10203429, -0.01565128],
        [-0.00743966, -0.05111152, -0.00687465,  0.02170176]],
       dtype=float32),
 array([ 0.06558485, -0.00913769, -0.01433168,  0.00298967], dtype=float32),
 array([[-0.0308837 ],
        [-0.04596547],
        [ 0.0354474 ],
        [ 0.03331755]], dtype=float32),
 array([-0.28104028], dtype=float32)]

In [69]:
nn_model_copy.get_weights()

[array([[-0.01507661, -0.00231922,  0.04057188, -0.05073781],
        [ 0.01343961,  0.00701454, -0.02576511,  0.00464113],
        [-0.09296036,  0.08705455, -0.05997289, -0.01714684],
        [-0.01427092, -0.01460152,  0.03518709,  0.02020625]],
       dtype=float32),
 array([0.05875353, 0.02737204, 0.02773035, 0.00149407], dtype=float32),
 array([[-0.05968304],
        [ 0.02631331],
        [-0.12372439],
        [ 0.05762375]], dtype=float32),
 array([0.04384924], dtype=float32)]

In [72]:
print(f"expected: {y_expected_0}, y_predicted: {nn_model(x_0)}")

expected: [[0.5]], y_predicted: [[-0.27458546]]


In [73]:
nn_model(x_0)

<tf.Tensor: shape=(1, 1), dtype=float32, numpy=array([[-0.27458546]], dtype=float32)>

In [75]:
for i in range(100):
    adjust_weights(x_0, y_expected_0)

print(f"iteration {i} ({y_expected_0}): {nn_model(x_0)}")

for i in range(100):
    adjust_weights(x_1, y_expected_1)

print(f"iteration {i} ({y_expected_1}): {nn_model(x_1)}")

print(f"Test: {x_0} -> {nn_model(x_0)}, {x_1} -> {nn_model(x_1)}")

iteration 99 ([[0.5]]): [[0.20597108]]
iteration 99 ([[-0.5]]): [[-0.23267356]]
Test: [[1. 1. 1. 1.]] -> [[-0.2325939]], [[-1.  1. -1.  1.]] -> [[-0.23267356]]


In [84]:
for i in range(1000):
    adjust_weights(x_0, y_expected_0)
    adjust_weights(x_1, y_expected_1)

print(f"both iterations {i} ({y_expected_0}, {y_expected_1}): {nn_model(x_0)}, {nn_model(x_1)}")

print(f"Test: {x_0} -> {nn_model(x_0)}, {x_1} -> {nn_model(x_1)}")

both iterations 999 ([[0.5]], [[-0.5]]): [[-0.00257209]], [[0.00838204]]
Test: [[1. 1. 1. 1.]] -> [[-0.00257209]], [[-1.  1. -1.  1.]] -> [[0.00838204]]


In [88]:
nn_model.compile(optimizer=tf.keras.optimizers.SGD(learning_rate=0.01), loss=keras.losses.mean_squared_error)

x_data = tf.constant([[1., 1., 1., 1.], [-1., 1., -1., 1.]])
y_data = tf.constant([[0.5], [-0.5]])

nn_model.fit(x_data, y_data, epochs=100, verbose=0)

In [90]:
nn_model.fit(x_data, y_data, epochs=1000, verbose=0)

print(f"Test: {x_0} -> {nn_model(x_0)}, {x_1} -> {nn_model(x_1)}")

Test: [[1. 1. 1. 1.]] -> [[0.4995811]], [[-1.  1. -1.  1.]] -> [[-0.49986598]]
